# 삼성전자 하루 방향 예측: ML · Transformer · 금융 시계열 파운데이션 모델 비교

이 노트북은 **Google Colab에서 위에서부터 순서대로 실행**하도록 만들었습니다. 삼성전자(005930)의 당일 종가가 전일 종가보다 얼마나 움직였는지를 다음 세 클래스로 예측합니다.

- **하락(0)**: 수익률 < -band
- **보합(1)**: -band ≤ 수익률 ≤ +band
- **상승(2)**: 수익률 > +band

`band`는 기본적으로 **최근 20일 변동성의 0.3배**(변동성 스케일 밴드)이며, 설정에서 고정 ±0.5%로 바꿀 수 있습니다. 타깃도 종가→종가 또는 시가→종가(장중 수익률) 중에서 고를 수 있습니다.

> ## ⚠️ 이 노트북이 실제로 예측하는 것
>
> 감사 결과, 종가→종가 타깃에서 측정되는 예측력의 대부분은 **전일 종가 → 당일 시가 "갭"** 입니다.
> 갭 부호 판별 AUC는 약 **0.76**인 반면, 시가에 진입해 종가에 청산하는 **세션 구간의 AUC는 약 0.52**로
> 사실상 무작위입니다. 갭은 09:00 시가에 이미 가격에 반영되므로 07:00 예측으로는 취할 수 없습니다.
>
> 그래서 8절에 **갭/세션 분해 표**를 두어, 매 실행마다 "거래 가능한 예측력"이 얼마인지 직접 보이도록 했습니다.
> balanced accuracy만 보고 성능을 판단하지 마세요 — 이 지표는 `VOL_BAND_MULT`를 키우면 단조 증가합니다.

동일한 시간 순서 검증에서 다음 방법을 비교합니다.

1. 항상 보합을 예측하는 기준선(클래스 사전확률)
2. 다항 로지스틱 회귀
3. LightGBM
4. 직접 학습하는 작은 Transformer Encoder (`RUN_TRANSFORMER`)
5. 금융 OHLCV 전용 시계열 파운데이션 모델 **Kronos-small**(zero-shot, `RUN_KRONOS`, 기본 꺼짐)
6. 위 모델 확률의 **단순 평균 앙상블**

> 중요: 주가 방향은 잡음이 매우 큰 문제입니다. 이 노트북은 투자수익을 보장하지 않으며, **어떤 모델이 동일 조건의 과거 외삽 검증에서 상대적으로 나았는지** 확인하기 위한 연구용 도구입니다.

### 누수 방지 원칙

- 삼성전자·KOSPI 지표는 예측일의 전 거래일까지만 사용합니다.
- 한국 지수는 **삼성 거래일 달력에 먼저 정렬한 뒤** 차분합니다(시계열 구멍이 다기간 수익률을 1일 수익률로 둔갑시키는 것을 막습니다).
- 미국시장 자료는 미국 세션 날짜에 하루를 더해 한국 아침에 도착한 것으로 정렬합니다.
- 아직 마감되지 않은 당일 봉은 자산군별 마감 시각을 기준으로 제거합니다.
- 데이터는 무작위로 섞지 않고 과거로 학습한 뒤 미래 구간을 예측합니다.
- 모든 모델 비교에 **월 블록 부트스트랩 신뢰구간**을 붙입니다. CI가 0을 포함하면 "동률"입니다.

## 0. Colab 런타임

`런타임 → 런타임 유형 변경 → T4 GPU`를 권장합니다. `QUICK_MODE=True`이면 Transformer와 Kronos 평가 구간을 줄여 빠르게 확인합니다. 전체 검증은 설정 셀에서 `QUICK_MODE=False`로 바꾸세요.


In [ ]:
%%capture
# 숫자에 직접 영향을 주는 두 패키지만 버전을 고정한다.
#   - lightgbm: 빌드가 다르면 같은 시드에서도 폴드별 balanced accuracy가 0.07까지 움직인다.
#   - yfinance: 데이터 스키마와 조정계수 처리가 버전마다 바뀐다.
# 나머지는 Colab 런타임 버전을 쓰되, 실제 사용된 버전을 config.json에 기록한다.
# (Kronos 관련 설치는 RUN_KRONOS=True일 때 7절에서만 수행한다.)
!pip -q install "yfinance==1.7.0" "lightgbm==4.7.0" seaborn joblib pyarrow exchange_calendars

In [ ]:
import os
import sys
import gc
import json
import math
import random
import time
import hashlib
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import sklearn
import lightgbm as lgb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
    confusion_matrix,
)
from lightgbm import LGBMClassifier, LGBMRegressor
import joblib

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except Exception as exc:  # torch 없는 CPU 전용 환경도 지원한다.
    torch = None
    TORCH_AVAILABLE = False
    print("torch를 불러오지 못했습니다. Transformer/Kronos 섹션은 건너뜁니다:", exc)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    DEVICE = "cpu"

# 실행 환경을 기록해 두면 나중에 숫자를 재현할 수 있다.
VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "lightgbm": lgb.__version__,
    "yfinance": yf.__version__,
    "torch": torch.__version__ if TORCH_AVAILABLE else None,
}
print(json.dumps(VERSIONS, indent=2))
print("Device:", DEVICE)
if TORCH_AVAILABLE and getattr(DEVICE, "type", "cpu") != "cuda":
    print("CPU에서도 실행되지만 Transformer/Kronos는 느립니다. Colab GPU를 권장합니다.")

## 1. 실험 설정

처음에는 아래 기본값으로 실행하는 것을 권장합니다. `KRONOS_EVAL_DAYS`는 파운데이션 모델의 과거 예측 횟수이며, 크게 잡을수록 훨씬 오래 걸립니다.


In [ ]:
START_DATE = "2015-01-01"
END_DATE = None                 # None이면 현재까지
NEUTRAL_BAND = 0.005            # BAND_MODE="fixed"일 때 ±0.5%

# --- 타깃 정의 ---
# TARGET_MODE
#   "close_to_close": 전일 종가 대비 당일 종가 수익률 (07:00 예측)
#                     ⚠️ 이 수익률의 약 48%는 "전일 종가→당일 시가" 갭이며, 갭은 09:00에
#                     이미 가격에 반영되어 있어 07:00 예측으로는 취할 수 없다. 8절의
#                     갭/세션 분해 표에서 실제로 거래 가능한 예측력을 확인하라.
#   "open_to_close" : 당일 시가 대비 당일 종가 수익률 (09:00 시가 확정 후 예측).
#                     실제로 거래 가능한 타깃이지만, 검증 결과 이 구간의 예측력은
#                     클래스 사전확률과 통계적으로 구분되지 않는다.
TARGET_MODE = "close_to_close"
# BAND_MODE
#   "fixed"     : ±NEUTRAL_BAND 고정
#   "vol_scaled": ±VOL_BAND_MULT × 최근 20일 일간수익률 표준편차(전일까지).
# ⚠️ balanced accuracy는 밴드 폭에 따라 단조 증가하므로, 이 값을 balanced accuracy로
#    튜닝하지 말 것. 왕복 거래비용(약 0.2~0.3%) 기준으로 고정하는 것이 옳다.
BAND_MODE = "vol_scaled"
VOL_BAND_MULT = 0.3
# open_to_close 모드에서 예측일 09:00 시가를 직접 입력한다.
# None이면 라이브 예측을 만들지 않고 중단한다(과거 백테스트는 정상 수행).
LIVE_OPEN_PRICE = None

FIRST_TEST_DATE = "2021-01-01"
TEST_MONTHS = 6                 # 워크포워드 한 구간의 길이
ROLLING_TRAIN_YEARS = 5         # 각 구간에서 최근 5년만 학습

QUICK_MODE = False
MAX_FOLDS = 3 if QUICK_MODE else None

# --- 모델 구성 ---
# 라이브 앙상블은 검증에서 가장 좋았던 조합(정규화한 Logistic + 작은 LightGBM의 단순 평균)을
# 사용한다. 스태킹 메타모델과 log loss 기반 지수가중은 검증에서 단순 평균보다 나빴으므로
# 제거했다.
ENSEMBLE_MODELS = ["Logistic", "LightGBM"]

RUN_TRANSFORMER = True          # 비교용. GPU가 없으면 False로 두면 훨씬 빠르다.
SEQ_LEN = 30
TRANSFORMER_EPOCHS = 8 if QUICK_MODE else 25
TRANSFORMER_PATIENCE = 3 if QUICK_MODE else 5
BATCH_SIZE = 64

# Kronos는 zero-shot 성능이 "항상 보합" 기준선보다 나빴고(log loss 1.87 vs 1.10),
# 무거운 의존성(git clone + HF 가중치)의 유일한 원인이므로 기본값을 False로 둔다.
RUN_KRONOS = False
KRONOS_LOOKBACK = 400
KRONOS_EVAL_DAYS = 20 if QUICK_MODE else 60
KRONOS_MC_SAMPLES = 3 if QUICK_MODE else 8

# --- 평가 설정 ---
COST_BP = 20.0                  # 왕복 거래비용(bp). 한국 단일종목은 매도 거래세 0.15% 포함.
BOOTSTRAP_B = 2000 if not QUICK_MODE else 400   # 월 블록 부트스트랩 반복수
DATA_CACHE_DIR = "/content/data_cache"          # 원본 시세 스냅샷 저장 위치
USE_DATA_CACHE = True           # True면 캐시가 있을 때 재다운로드하지 않는다.

# 한국 휴일 때문에 자동 계산 날짜가 틀리면 "2026-09-08"처럼 직접 지정하세요.
PREDICTION_DATE_OVERRIDE = None

LABEL_NAMES = {0: "하락", 1: "보합", 2: "상승"}
PROB_COLS = ["p_down", "p_flat", "p_up"]
# 표에서 모델 순서를 고정한다(성능순 정렬은 잡음을 순위로 보이게 만든다).
MODEL_ORDER = ["Always flat", "Logistic", "LightGBM", "Transformer", "Mean ensemble", "Kronos-small"]

print({
    "QUICK_MODE": QUICK_MODE,
    "TARGET_MODE": TARGET_MODE,
    "BAND_MODE": BAND_MODE,
    "ENSEMBLE_MODELS": ENSEMBLE_MODELS,
    "RUN_TRANSFORMER": RUN_TRANSFORMER,
    "RUN_KRONOS": RUN_KRONOS,
    "COST_BP": COST_BP,
})

## 2. 데이터 다운로드

Colab에서 별도 API 키 없이 재현할 수 있도록 Yahoo Finance를 사용합니다. 일부 티커가 일시적으로 실패해도 나머지 데이터로 계속 진행합니다.

**예측일 오전 7시까지 사용할 신호**

- 삼성전자·KOSPI·KOSPI200·SK하이닉스의 과거 수익률, 추세, 변동성, 거래량
- 런던 삼성전자 GDR(SMSN.IL): 한국 장 마감 후 거래되는 삼성전자 가격의 대리변수
- 요일, 월말 여부 등 달력 변수
- SOX, Nasdaq, S&P 500, Micron, Nvidia, TSMC ADR, EWY
- 원/달러, 달러지수, VIX, 미국 10년물 금리, WTI

Yahoo Finance는 연구·교육용 편의 데이터입니다. 실제 운영에서는 KRX, 한국은행 ECOS, CME 등 원천 자료로 교체하는 것이 좋습니다.


In [ ]:
# KOSPI200(^KS200)은 KOSPI와 사실상 중복이면서(ret_1 상관 0.964, vol_20 상관 0.996)
# Yahoo 시계열에 긴 공백이 생기는 일이 잦아, 그 공백이 전체 학습 구간을 잘라내고
# 라이브 피처를 오염시켰다. 그래서 자산 목록에서 제외했다.
ASSETS = {
    "samsung": "005930.KS",
    "kospi": "^KS11",
    "sk_hynix": "000660.KS",
    "samsung_gdr": "SMSN.IL",
    "sox": "^SOX",
    "nasdaq": "^IXIC",
    "sp500": "^GSPC",
    "micron": "MU",
    "nvidia": "NVDA",
    "tsmc_adr": "TSM",
    "korea_etf": "EWY",
    "usdkrw": "KRW=X",
    "dxy": "DX-Y.NYB",
    "vix": "^VIX",
    "us10y": "^TNX",
    "wti": "CL=F",
}

# 자산군별 "마지막 봉이 마감되었다고 볼 수 있는 시각". 이보다 이른 시각에 실행하면
# Yahoo가 아직 진행 중인 당일 봉을 돌려주므로, 그 봉은 버린다.
KOREAN_ASSETS = ["samsung", "kospi", "sk_hynix"]
GLOBAL_ASSETS = [
    "sox", "nasdaq", "sp500", "micron", "nvidia", "tsmc_adr",
    "korea_etf", "usdkrw", "dxy", "vix", "us10y", "wti", "samsung_gdr",
]
SESSION_CLOSE = {           # (tz, 마감 시각) — 이 시각 이전이면 당일 봉을 미완성으로 본다.
    "korea":  ("Asia/Seoul", 15, 40),
    "us":     ("America/New_York", 16, 5),
    "london": ("Europe/London", 16, 35),
    "cont":   ("America/New_York", 17, 5),   # FX·선물 등 사실상 24시간 시장
}
ASSET_SESSION = {name: "korea" for name in KOREAN_ASSETS}
ASSET_SESSION.update({n: "us" for n in ["sox", "nasdaq", "sp500", "micron", "nvidia", "tsmc_adr", "korea_etf", "us10y"]})
ASSET_SESSION["samsung_gdr"] = "london"
ASSET_SESSION.update({n: "cont" for n in ["usdkrw", "dxy", "vix", "wti"]})


def _flatten_yf_columns(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame.columns = [str(c).strip().lower().replace(" ", "_") for c in frame.columns]
    return frame


def drop_unclosed_last_bar(frame, session):
    """아직 마감되지 않은 당일 봉을 제거한다(학습/서빙 분포 불일치 방지)."""
    if frame.empty:
        return frame
    tz, hour, minute = SESSION_CLOSE[session]
    now_local = pd.Timestamp.now(tz=tz)
    last_date = frame.index.max().date()
    if last_date >= now_local.date() and (now_local.hour, now_local.minute) < (hour, minute):
        return frame.iloc[:-1].copy()
    return frame


def find_placeholder_bars(frame):
    """거래량 0에 시가=고가=저가=종가인 Yahoo 유령봉(실제로는 거래가 있었던 날)."""
    if frame.empty or "volume" not in frame:
        return pd.Series(False, index=frame.index)
    return (frame["volume"] == 0) & (frame["high"] == frame["low"]) & (frame["open"] == frame["close"])


def drop_placeholder_bars(frame, name):
    """삼성전자만 실제로 제거한다.
    유령봉은 target_return을 정확히 0으로 만들어 '보합' 라벨을 조작하므로 타깃 자산에서는
    반드시 빼야 한다. 반면 보조 자산에서 빼면 그 날짜가 NaN이 되어 dropna가 멀쩡한
    삼성 관측치까지 학습에서 날려버리므로(약 100행), 개수만 보고하고 그대로 둔다."""
    bad = find_placeholder_bars(frame)
    n = int(bad.sum())
    if n == 0:
        return frame, []
    dates = [d.date().isoformat() for d in frame.index[bad]]
    if name == "samsung":
        return frame[~bad].copy(), dates
    return frame, dates


def download_one(ticker, start=START_DATE, end=END_DATE, retries=3):
    last_error = None
    for attempt in range(retries):
        try:
            frame = yf.download(
                ticker, start=start, end=end, auto_adjust=False,
                progress=False, threads=False, timeout=30,
            )
            frame = _flatten_yf_columns(frame)
            if frame.empty:
                raise ValueError("empty response")
            frame.index = pd.to_datetime(frame.index)
            if frame.index.tz is not None:
                frame.index = frame.index.tz_localize(None)
            frame.index = frame.index.normalize()
            frame = frame[~frame.index.duplicated(keep="last")].sort_index()
            if "adj_close" not in frame.columns and "close" in frame.columns:
                frame["adj_close"] = frame["close"]
            keep = [c for c in ["open", "high", "low", "close", "adj_close", "volume"] if c in frame]
            frame = frame[keep].astype(float)
            # Yahoo가 조정계수를 float32로 반올림해 재다운로드마다 미세하게 값이 달라진다.
            # 유효숫자 8자리로 잘라 실행 간 재현성을 확보한다.
            for col in ["open", "high", "low", "close", "adj_close"]:
                if col in frame:
                    frame[col] = frame[col].round(6)
            return frame
        except Exception as exc:
            last_error = exc
            time.sleep(1.5 * (attempt + 1))
    print(f"⚠️ {ticker} 다운로드 실패: {last_error}")
    return pd.DataFrame()


def _cache_write(frame, cache, name):
    """parquet을 우선 쓰되, 엔진이 없으면 CSV로 떨어뜨린다."""
    try:
        frame.to_parquet(cache / f"{name}.parquet")
    except Exception:
        frame.to_csv(cache / f"{name}.csv")


def _cache_read(cache, name):
    parquet, csv = cache / f"{name}.parquet", cache / f"{name}.csv"
    if parquet.exists():
        try:
            return pd.read_parquet(parquet)
        except Exception:
            pass
    if csv.exists():
        return pd.read_csv(csv, index_col=0, parse_dates=True)
    return None


def load_raw(assets, cache_dir=DATA_CACHE_DIR, use_cache=USE_DATA_CACHE):
    """캐시가 있으면 재사용하고, 없으면 내려받아 저장한다(스냅샷 재현성).
    같은 날 저녁에 다시 받기만 해도 Yahoo 값이 미세하게 달라져 백테스트가 흔들리므로,
    한 번 받은 원본을 파일로 남겨 두는 것이 재현의 전제다."""
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    out, from_cache = {}, []
    for name, ticker in tqdm(assets.items(), desc="Loading"):
        if use_cache:
            cached = _cache_read(cache, name)
            if cached is not None:
                out[name] = cached
                from_cache.append(name)
                continue
        frame = download_one(ticker)
        frame = drop_unclosed_last_bar(frame, ASSET_SESSION.get(name, "cont"))
        frame, flagged = drop_placeholder_bars(frame, name)
        if flagged:
            action = "제거" if name == "samsung" else "발견(유지)"
            print(f"  {name}: 유령봉 {len(flagged)}개 {action} (예: {flagged[:3]})")
        if not frame.empty:
            _cache_write(frame, cache, name)
        out[name] = frame
    if from_cache:
        print(f"캐시에서 로드: {len(from_cache)}개 ({cache}). 새로 받으려면 USE_DATA_CACHE=False.")
    return out


raw = load_raw(ASSETS)

if raw["samsung"].empty:
    raise RuntimeError("삼성전자 데이터 다운로드에 실패했습니다. 잠시 후 셀을 다시 실행하세요.")

# ---- 자산별 신선도·공백 검증 -------------------------------------------------
# 하나의 보조 시계열에 구멍이 나면 dropna가 최근 구간을 통째로 날리고, 라이브 행에는
# 며칠~몇 주짜리 이동이 "1일 수익률"로 들어간다. 그래서 여기서 먼저 걸러낸다.
MAX_STALE_DAYS = 7            # 마지막 봉이 삼성 마지막 거래일보다 이만큼 오래되면 제외
MAX_MISSING_RECENT = 5        # 최근 250거래일 중 결측 허용 개수

sam_calendar = raw["samsung"].index
last_samsung_date = sam_calendar.max()
quality, dropped_assets = [], []
for name, frame in list(raw.items()):
    if frame.empty:
        dropped_assets.append((name, "다운로드 실패"))
        continue
    recent = sam_calendar[-250:]
    missing = len(recent.difference(frame.index)) if name in KOREAN_ASSETS else 0
    stale_days = (last_samsung_date - frame.index.max()).days
    reason = None
    if name != "samsung":
        if stale_days > MAX_STALE_DAYS:
            reason = f"마지막 봉이 {stale_days}일 전"
        elif missing > MAX_MISSING_RECENT:
            reason = f"최근 250거래일 중 {missing}일 결측"
    quality.append({
        "asset": name, "ticker": ASSETS[name], "rows": len(frame),
        "first": frame.index.min(), "last": frame.index.max(),
        "stale_days": stale_days, "missing_recent": missing,
        "status": "제외됨" if reason else "정상",
    })
    if reason:
        dropped_assets.append((name, reason))
        raw.pop(name)

for name, reason in dropped_assets:
    print(f"⚠️ {name} 제외: {reason}")

DATA_SNAPSHOT_HASH = hashlib.sha256(
    "|".join(f"{n}:{len(f)}:{f.index.max().date()}" for n, f in sorted(raw.items())).encode()
).hexdigest()[:16]
print("데이터 스냅샷 해시:", DATA_SNAPSHOT_HASH)

pd.DataFrame(quality)

## 3. 특징 생성과 시각 정렬

행의 날짜가 예측 대상 거래일입니다. 삼성전자와 한국시장 신호에는 `shift(1)`을 적용해 전 거래일까지만 사용합니다. 미국시장 날짜 `d`의 종가는 한국시간으로 다음 날 새벽에 확정되므로 가용 날짜를 `d+1일`로 옮긴 뒤 가장 최근 자료를 결합합니다. 주말은 이전 값을 이어받되 미래 값으로 뒤를 채우지 않습니다.


In [ ]:
def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def rolling_zscore(series, window=20):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()
    return (series - mean) / std.replace(0, np.nan)


def safe_pct_change(close, periods=1, max_gap_days=None):
    """시계열에 구멍이 있으면 다기간 수익률이 1일 수익률로 둔갑한다.
    직전 관측이 너무 오래되었으면 NaN으로 만들어 결측으로 드러나게 한다."""
    if max_gap_days is None:
        max_gap_days = 4 * periods + 3
    ret = close.pct_change(periods)
    elapsed = close.index.to_series().diff(periods).dt.days
    return ret.where(elapsed <= max_gap_days)


def _as_ns(index):
    """merge_asof는 양쪽 키의 datetime 단위가 같아야 한다(pandas 3.0에서 us/ns가 섞인다)."""
    return pd.DatetimeIndex(pd.to_datetime(index)).as_unit("ns")


def merge_latest_available(base_index, series, availability_days=1):
    values = series.dropna().copy()
    values.index = (_as_ns(values.index).tz_localize(None).normalize()
                    + pd.Timedelta(days=availability_days))
    values = values.groupby(level=0).last().sort_index()
    left = pd.DataFrame({"date": _as_ns(base_index)}).sort_values("date")
    right = values.rename("value").reset_index()
    right.columns = ["available_date", "value"]
    merged = pd.merge_asof(
        left, right.sort_values("available_date"),
        left_on="date", right_on="available_date",
        direction="backward", tolerance=pd.Timedelta(days=7),
    )
    return pd.Series(merged["value"].to_numpy(), index=left["date"], name=series.name)


def next_krx_session(after):
    """다음 KRX 거래일. exchange_calendars가 있으면 공휴일까지 반영한다."""
    try:
        import exchange_calendars as xcals
        cal = xcals.get_calendar("XKRX")
        return pd.Timestamp(cal.next_session(pd.Timestamp(after))).normalize()
    except Exception:
        print("exchange_calendars 없음: 주말만 제외해 예측일을 계산합니다(공휴일 미반영).")
        return pd.bdate_range(pd.Timestamp(after) + pd.Timedelta(days=1), periods=1)[0]


def krx_sessions_ahead(start, n_sessions):
    """start(포함)부터 n_sessions번째 KRX 거래일."""
    try:
        import exchange_calendars as xcals
        cal = xcals.get_calendar("XKRX")
        return pd.Timestamp(cal.sessions_window(pd.Timestamp(start), n_sessions - 1)[-1]).normalize()
    except Exception:
        return pd.bdate_range(pd.Timestamp(start), periods=n_sessions)[-1]


sam = raw["samsung"].copy()
sam = sam.dropna(subset=["open", "high", "low", "close", "adj_close"])
last_samsung_date = sam.index.max()

if PREDICTION_DATE_OVERRIDE:
    prediction_date = pd.Timestamp(PREDICTION_DATE_OVERRIDE).normalize()
else:
    prediction_date = next_krx_session(last_samsung_date)

all_dates = sam.index.union(pd.DatetimeIndex([prediction_date])).sort_values()
feat = pd.DataFrame(index=all_dates)

sam_close = sam["adj_close"]          # 타깃과 수익률 계열은 배당조정 종가 기준
sam_raw_close = sam["close"]          # 갭/세션 분해와 표시 가격은 원본 종가 기준
sam_ret = sam_close.pct_change()
sam_gap = sam["open"] / sam_raw_close.shift(1) - 1
sam_range = (sam["high"] - sam["low"]) / sam_raw_close
sam_log_volume = np.log1p(sam["volume"].clip(lower=0))

# 예측일에는 전 거래일까지 확정된 값만 들어간다.
feat["sam_ret_1"] = sam_ret.reindex(all_dates).shift(1)
feat["sam_ret_2"] = sam_close.pct_change(2).reindex(all_dates).shift(1)
feat["sam_ret_5"] = sam_close.pct_change(5).reindex(all_dates).shift(1)
feat["sam_ret_20"] = sam_close.pct_change(20).reindex(all_dates).shift(1)
feat["sam_vol_5"] = sam_ret.rolling(5).std().reindex(all_dates).shift(1)
feat["sam_vol_20"] = sam_ret.rolling(20).std().reindex(all_dates).shift(1)
feat["sam_gap_1"] = sam_gap.reindex(all_dates).shift(1)
feat["sam_range_1"] = sam_range.reindex(all_dates).shift(1)
feat["sam_volume_z20"] = rolling_zscore(sam_log_volume, 20).reindex(all_dates).shift(1)
feat["sam_price_to_ma20"] = (sam_close / sam_close.rolling(20).mean() - 1).reindex(all_dates).shift(1)
feat["sam_price_to_ma60"] = (sam_close / sam_close.rolling(60).mean() - 1).reindex(all_dates).shift(1)
feat["sam_rsi14"] = (rsi(sam_close, 14) / 100).reindex(all_dates).shift(1)

# 한국시장: 같은 날짜의 종가는 오전 7시에 아직 없으므로 1거래일 지연.
# ⚠️ 자산 고유 인덱스가 아니라 "삼성 거래일 달력"에 먼저 맞춘 뒤 차분한다.
#    그렇지 않으면 시계열 구멍이 다기간 수익률을 1일 수익률로 둔갑시킨다.
for name in ["kospi", "sk_hynix"]:
    frame = raw.get(name, pd.DataFrame())
    if frame.empty or "adj_close" not in frame:
        continue
    close = frame["adj_close"].reindex(sam.index)
    feat[f"{name}_ret_1"] = safe_pct_change(close, 1).reindex(all_dates).shift(1)
    feat[f"{name}_ret_5"] = safe_pct_change(close, 5).reindex(all_dates).shift(1)
    feat[f"{name}_vol_20"] = safe_pct_change(close, 1).rolling(20).std().reindex(all_dates).shift(1)

# 해외시장: 미국 세션 d의 정보는 한국 날짜 d+1 아침에 사용 가능하다고 보수적으로 정렬.
for name in GLOBAL_ASSETS:
    frame = raw.get(name, pd.DataFrame())
    if frame.empty or "adj_close" not in frame:
        continue
    close = frame["adj_close"]
    feat[f"{name}_ret_1"] = merge_latest_available(all_dates, safe_pct_change(close, 1), 1)
    feat[f"{name}_ret_5"] = merge_latest_available(all_dates, safe_pct_change(close, 5), 1)
    if name in ["vix", "us10y", "usdkrw", "dxy", "wti"]:
        feat[f"{name}_level_z60"] = merge_latest_available(all_dates, rolling_zscore(close, 60), 1)

# 런던 GDR 세션 d 수익률 - 삼성전자 세션 d 수익률: 한국 장 마감 이후의 가격 변화(야간 신호).
if "samsung_gdr_ret_1" in feat:
    feat["gdr_overnight_signal"] = feat["samsung_gdr_ret_1"] - feat["sam_ret_1"]

# 달력 변수: 예측일 자체의 속성이므로 shift가 필요 없다.
feat["cal_dow"] = feat.index.dayofweek.astype(float)
feat["cal_is_monday"] = (feat.index.dayofweek == 0).astype(float)
feat["cal_is_friday"] = (feat.index.dayofweek == 4).astype(float)
feat["cal_month_end"] = (feat.index.day >= 26).astype(float)

# 발표 빈도가 다른 자료를 합칠 때 미래값으로 backfill하지 않는다.
feat = feat.replace([np.inf, -np.inf], np.nan).ffill(limit=5)

# 시가→종가 모드: 당일 시가는 09:00에 확정되므로 당일 갭을 특징으로 쓸 수 있다.
if TARGET_MODE == "open_to_close":
    feat["sam_gap_0"] = sam_gap.reindex(all_dates)

if TARGET_MODE == "open_to_close":
    target_return = (sam_raw_close / sam["open"] - 1).reindex(all_dates)
elif TARGET_MODE == "close_to_close":
    target_return = sam_close.pct_change().reindex(all_dates)
else:
    raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")

# 보합 밴드. vol_scaled는 전일까지의 변동성만 사용하므로 예측 시점에 알 수 있는 값이다.
if BAND_MODE == "vol_scaled":
    band_source = target_return if TARGET_MODE == "open_to_close" else sam_ret
    band = (VOL_BAND_MULT * band_source.rolling(20).std()).reindex(all_dates).shift(1)
elif BAND_MODE == "fixed":
    band = pd.Series(NEUTRAL_BAND, index=all_dates, dtype=float)
else:
    raise ValueError(f"Unknown BAND_MODE: {BAND_MODE}")
feat["band"] = band


def label_from_return(ret, band_value):
    """수익률과 밴드로부터 3-class 라벨. 사후 채점에서도 같은 함수를 쓴다."""
    ret = np.asarray(ret, dtype=float)
    band_value = np.asarray(band_value, dtype=float)
    out = np.select([ret < -band_value, ret > band_value], [0, 2], default=1).astype(float)
    out[np.isnan(ret) | np.isnan(band_value)] = np.nan
    return out


feat["target_return"] = target_return
feat["target"] = label_from_return(target_return.to_numpy(), band.to_numpy())

# 갭 / 세션 분해 — 평가에서만 쓰며 특징으로는 절대 넣지 않는다.
# ⚠️ 원본 close 기준이어야 (1+gap)(1+session)이 원본 종가→종가 수익률과 일치한다.
decomp = pd.DataFrame({
    "gap": (sam["open"] / sam_raw_close.shift(1) - 1),
    "session": (sam_raw_close / sam["open"] - 1),
})

# 결측이 지나치게 많은 특징은 제거한다.
NON_FEATURE_COLS = ["target", "target_return", "band"]
candidate_features = [c for c in feat.columns if c not in NON_FEATURE_COLS]
coverage = feat.loc[sam.index, candidate_features].notna().mean()
feature_cols = coverage[coverage >= 0.80].index.tolist()
removed_features = coverage[coverage < 0.80].index.tolist()

model_df = feat.loc[sam.index, feature_cols + NON_FEATURE_COLS].dropna().copy()
model_df["target"] = model_df["target"].astype(int)

# ---- 잘려나간 최근 구간이 있는지 반드시 확인한다 ----------------------------
trailing = feat.loc[sam.index].loc[model_df.index.max():].iloc[1:]
if len(trailing):
    blocking = (
        feat.loc[trailing.index, feature_cols].isna().sum()
        .sort_values(ascending=False).head(5)
    )
    print(f"⚠️ dropna로 최근 {len(trailing)}행이 제외되었습니다. 원인 컬럼:")
    print(blocking[blocking > 0].to_string())
staleness_days = (last_samsung_date - model_df.index.max()).days
assert staleness_days <= 10, (
    f"학습 데이터가 {staleness_days}일 뒤처져 있습니다. 위 원인 컬럼을 제거하거나 보정하세요."
)

live_row = feat.loc[[prediction_date], feature_cols].copy()
live_band = float(feat.loc[prediction_date, "band"]) if pd.notna(feat.loc[prediction_date, "band"]) else float(model_df["band"].iloc[-1])
if TARGET_MODE == "open_to_close":
    if LIVE_OPEN_PRICE is None:
        raise ValueError(
            "TARGET_MODE='open_to_close'에서는 예측일 09:00 시가(LIVE_OPEN_PRICE)가 필요합니다. "
            "갭을 0으로 가정하면 백테스트가 검증한 것과 다른 예측이 됩니다."
        )
    live_row.loc[prediction_date, "sam_gap_0"] = float(LIVE_OPEN_PRICE) / float(sam_raw_close.iloc[-1]) - 1

# 라이브 행의 결측을 과거값으로 메울 때는 무엇을 언제 값으로 메웠는지 반드시 남긴다.
imputed_live_features = {}
for col in feature_cols:
    if pd.isna(live_row.iloc[0][col]):
        source_date = model_df[col].last_valid_index()
        live_row.loc[prediction_date, col] = model_df[col].iloc[-1]
        imputed_live_features[col] = str(source_date.date()) if source_date is not None else None
if imputed_live_features:
    print("⚠️ 라이브 행에서 과거값으로 대체된 특징:", imputed_live_features)

# 라이브 특징이 학습 분포를 크게 벗어나면 데이터 문제일 가능성이 높다.
lo = model_df[feature_cols].quantile(0.001)
hi = model_df[feature_cols].quantile(0.999)
outliers = [c for c in feature_cols if not (lo[c] <= live_row.iloc[0][c] <= hi[c])]
if outliers:
    print("⚠️ 라이브 특징이 학습 분포의 0.1~99.9% 범위를 벗어남:", outliers)

print("Model rows:", len(model_df))
print("Features:", len(feature_cols))
print("Removed low-coverage features:", removed_features)
print("Backtest range:", model_df.index.min().date(), "~", model_df.index.max().date())
print("Last Samsung bar:", last_samsung_date.date(), f"(학습 데이터 지연 {staleness_days}일)")
print("Candidate prediction date:", prediction_date.date())
print("Target mode:", TARGET_MODE, "| Band mode:", BAND_MODE, "| Live band: ±{:.2%}".format(live_band))

assert prediction_date > last_samsung_date
assert model_df.index.is_monotonic_increasing
assert model_df[feature_cols].notna().all().all()

In [ ]:
class_counts = model_df["target"].value_counts().sort_index()
class_table = pd.DataFrame({
    "class": [LABEL_NAMES[i] for i in class_counts.index],
    "count": class_counts.values,
    "ratio": class_counts.values / class_counts.sum(),
}, index=class_counts.index)
display(class_table.style.format({"ratio": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_table.set_index("class")["count"].plot.bar(ax=axes[0], color=["#d95f5f", "#999999", "#4c78a8"])
axes[0].set_title("Target class distribution")
axes[0].set_xlabel("")
model_df["target_return"].clip(-0.08, 0.08).hist(bins=60, ax=axes[1], color="#4c78a8")
median_band = float(model_df["band"].median())
axes[1].axvline(-median_band, color="black", ls="--", label=f"median band ±{median_band:.2%}")
axes[1].axvline(median_band, color="black", ls="--")
axes[1].legend()
axes[1].set_title(f"Samsung {TARGET_MODE} return")
plt.tight_layout()
plt.show()


## 4. 워크포워드 검증과 공통 평가함수

각 폴드는 과거 5년을 학습하고 다음 6개월을 예측합니다. `QUICK_MODE=True`에서는 가장 최근 3개 폴드만 실행합니다. 모든 모델의 확률 열 순서는 `[하락, 보합, 상승]`으로 고정합니다.


In [ ]:
X = model_df[feature_cols].to_numpy(dtype=np.float32)
y = model_df["target"].to_numpy(dtype=np.int64)
dates = model_df.index
GAP = decomp["gap"].reindex(dates)
SESSION = decomp["session"].reindex(dates)


def make_walk_forward_folds(dates_index):
    folds = []
    start = pd.Timestamp(FIRST_TEST_DATE)
    max_date = dates_index.max()
    fold_no = 0
    while start <= max_date:
        end = start + pd.DateOffset(months=TEST_MONTHS)
        train_start = start - pd.DateOffset(years=ROLLING_TRAIN_YEARS)
        train_idx = np.where((dates_index >= train_start) & (dates_index < start))[0]
        test_idx = np.where((dates_index >= start) & (dates_index < end))[0]
        if len(train_idx) >= 500 and len(test_idx) >= 20:
            folds.append({
                "fold": fold_no, "train_idx": train_idx, "test_idx": test_idx,
                "train_start": dates_index[train_idx[0]], "train_end": dates_index[train_idx[-1]],
                "test_start": dates_index[test_idx[0]], "test_end": dates_index[test_idx[-1]],
            })
            fold_no += 1
        start = end
    if MAX_FOLDS is not None:
        folds = folds[-MAX_FOLDS:]
    return folds


folds = make_walk_forward_folds(dates)
if not folds:
    raise RuntimeError("워크포워드 폴드가 없습니다. START_DATE 또는 FIRST_TEST_DATE를 확인하세요.")

display(pd.DataFrame([{k: v for k, v in f.items() if not k.endswith("idx")} for f in folds]))


def multiclass_brier(y_true, probs):
    onehot = np.eye(3)[np.asarray(y_true, dtype=int)]
    return np.mean(np.sum((np.asarray(probs) - onehot) ** 2, axis=1))


def prediction_frame(model_name, row_dates, y_true, probs, fold_id=None, y_pred=None, extra=None):
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    probs = probs / probs.sum(axis=1, keepdims=True)
    out = pd.DataFrame({
        "date": pd.to_datetime(row_dates),
        "y_true": np.asarray(y_true, dtype=int),
        "y_pred": probs.argmax(axis=1) if y_pred is None else np.asarray(y_pred, dtype=int),
        "p_down": probs[:, 0], "p_flat": probs[:, 1], "p_up": probs[:, 2],
        "model": model_name, "fold": fold_id,
    })
    if extra:
        for key, value in extra.items():
            out[key] = value
    return out


# ---- 월 블록 부트스트랩 ------------------------------------------------------
# 일별 예측은 서로 독립이 아니므로, 달력 월을 블록으로 복원추출해 신뢰구간을 만든다.
def month_blocks(date_index):
    key = pd.PeriodIndex(pd.DatetimeIndex(date_index), freq="M")
    return [np.where(key == m)[0] for m in key.unique()]


def block_bootstrap_ci(date_index, stat_fn, b=BOOTSTRAP_B, seed=SEED, alpha=0.05):
    """stat_fn(row_positions) -> 스칼라. (하한, 상한) 반환."""
    blocks = month_blocks(date_index)
    if not blocks:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(b):
        pick = rng.integers(0, len(blocks), len(blocks))
        idx = np.concatenate([blocks[i] for i in pick])
        try:
            draws.append(stat_fn(idx))
        except Exception:
            continue
    if not draws:
        return (np.nan, np.nan)
    return tuple(np.percentile(draws, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


def metric_row(frame):
    true = frame["y_true"].to_numpy(dtype=int)
    pred = frame["y_pred"].to_numpy(dtype=int)
    probs = frame[PROB_COLS].to_numpy(dtype=float)
    return {
        "n": len(frame),
        "accuracy": accuracy_score(true, pred),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
        "macro_f1": f1_score(true, pred, average="macro", zero_division=0),
        "log_loss": log_loss(true, probs, labels=[0, 1, 2]),
        "brier": multiclass_brier(true, probs),
    }


def _auc_vs_sign(score, leg):
    """방향 점수가 특정 구간(갭 또는 세션)의 부호를 얼마나 맞히는가."""
    ok = leg.notna().to_numpy() & (leg.to_numpy() != 0)
    if ok.sum() < 30:
        return np.nan
    return roc_auc_score((leg.to_numpy()[ok] > 0).astype(int), np.asarray(score)[ok])


def decomposition_row(frame, cost_bp=COST_BP):
    """모델이 실제로 무엇을 예측하는지: 갭 vs 세션, 그리고 구간별 기대손익(bp)."""
    s = frame.set_index("date").sort_index()
    score = (s["p_up"] - s["p_down"]).to_numpy()
    gap = GAP.reindex(s.index)
    ses = SESSION.reindex(s.index)
    direction = np.sign(score)
    gap_bp = float(np.nanmean(direction * gap.to_numpy())) * 1e4
    ses_bp = float(np.nanmean(direction * ses.to_numpy())) * 1e4
    lo, hi = block_bootstrap_ci(
        s.index,
        lambda idx: float(np.nanmean(direction[idx] * ses.to_numpy()[idx])) * 1e4,
    )
    return {
        "auc_gap": _auc_vs_sign(score, gap),
        "auc_session": _auc_vs_sign(score, ses),
        "gap_bp": gap_bp,
        "session_bp": ses_bp,
        "session_bp_lo": lo,
        "session_bp_hi": hi,
        "session_bp_net": ses_bp - cost_bp,
    }


def summarize_predictions(predictions, with_ci=True, with_decomposition=False):
    rows = []
    for name, group in predictions.groupby("model"):
        row = {"model": name, **metric_row(group)}
        if with_ci:
            g = group.sort_values("date")
            true = g["y_true"].to_numpy(dtype=int)
            pred = g["y_pred"].to_numpy(dtype=int)
            probs = g[PROB_COLS].to_numpy(dtype=float)
            row["bal_acc_lo"], row["bal_acc_hi"] = block_bootstrap_ci(
                g["date"], lambda i: balanced_accuracy_score(true[i], pred[i]))
            row["log_loss_lo"], row["log_loss_hi"] = block_bootstrap_ci(
                g["date"], lambda i: log_loss(true[i], probs[i], labels=[0, 1, 2]))
        if with_decomposition:
            row.update(decomposition_row(group))
        rows.append(row)
    out = pd.DataFrame(rows).set_index("model")
    order = [m for m in MODEL_ORDER if m in out.index] + [m for m in out.index if m not in MODEL_ORDER]
    return out.loc[order]


def paired_delta_ci(predictions, model_a, model_b, metric="balanced_accuracy"):
    """같은 날짜에서 두 모델의 지표 차이와 95% 신뢰구간(월 블록 부트스트랩)."""
    a = predictions[predictions["model"] == model_a].set_index("date").sort_index()
    b = predictions[predictions["model"] == model_b].set_index("date").sort_index()
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    if len(common) < 60:
        return {"delta": np.nan, "lo": np.nan, "hi": np.nan, "n": len(common)}

    def stat(idx):
        fa, fb = a.iloc[idx], b.iloc[idx]
        if metric == "balanced_accuracy":
            return (balanced_accuracy_score(fa["y_true"], fa["y_pred"])
                    - balanced_accuracy_score(fb["y_true"], fb["y_pred"]))
        return (log_loss(fa["y_true"], fa[PROB_COLS].to_numpy(), labels=[0, 1, 2])
                - log_loss(fb["y_true"], fb[PROB_COLS].to_numpy(), labels=[0, 1, 2]))

    delta = stat(np.arange(len(common)))
    lo, hi = block_bootstrap_ci(common, stat)
    return {"delta": delta, "lo": lo, "hi": hi, "n": len(common)}


def folds_worse_than_prior(predictions, model_name):
    """폴드별 log loss가 '항상 보합'(클래스 사전확률)보다 나쁜 횟수."""
    g = predictions[predictions["model"] == model_name]
    base = predictions[predictions["model"] == "Always flat"].set_index(["fold", "date"])
    worse = 0
    for fold_id, grp in g.groupby("fold"):
        idx = pd.MultiIndex.from_arrays([grp["fold"], grp["date"]])
        try:
            ref = base.loc[idx]
        except KeyError:
            continue
        ll_model = log_loss(grp["y_true"], grp[PROB_COLS].to_numpy(), labels=[0, 1, 2])
        ll_base = log_loss(ref["y_true"], ref[PROB_COLS].to_numpy(), labels=[0, 1, 2])
        worse += int(ll_model >= ll_base)
    return worse

## 5. 기준선 · 로지스틱 회귀 · LightGBM

복잡한 모델은 반드시 단순 기준선보다 미래 구간에서 좋아야 합니다. 클래스 불균형을 고려해 로지스틱과 LightGBM 모두 균형 가중치를 사용합니다.


In [ ]:
# 셀을 다시 실행해도 예측 프레임이 중복 적재되지 않도록, 단계별 결과를 딕셔너리에
# 보관하고 매번 새로 조립한다.
PREDICTION_PARTS = {}


def assemble_predictions():
    parts = [p for p in PREDICTION_PARTS.values() if p is not None and len(p)]
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    assert not out.duplicated(["date", "model"]).any(), "예측 프레임에 중복 행이 있습니다."
    return out


def make_logistic():
    # C=0.25는 2026년 고변동 국면에서 과신이 심했다(마지막 폴드 log loss 1.25 > 사전확률 1.10).
    # C를 낮춰도 balanced accuracy는 그대로이고 확률 품질만 좋아진다.
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=0.01, max_iter=3000,
                                     class_weight="balanced", random_state=SEED)),
    ])


def make_lgbm(n_estimators=60, num_leaves=7):
    # 300트리 × 15리프는 그리드에서 가장 나쁜 조합이었고 클래스 사전확률보다도 log loss가
    # 높았다(1.099 vs 1.097). 60×7이 검증에서 최적이었다.
    # subsample은 subsample_freq를 켜지 않으면 아무 효과가 없으므로 아예 제거했다.
    return LGBMClassifier(
        objective="multiclass", num_class=3,
        n_estimators=n_estimators, learning_rate=0.03, num_leaves=num_leaves,
        min_child_samples=50, colsample_bytree=0.85,
        reg_alpha=0.5, reg_lambda=2.0, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )


tabular_predictions = []
lgb_importances = []

for fold in tqdm(folds, desc="Tabular walk-forward"):
    tr, te = fold["train_idx"], fold["test_idx"]

    # 기준선: 학습구간의 클래스 사전확률을 그대로 확률로 쓰고, 하드 예측만 보합으로 고정한다.
    # (확률을 왜곡해 argmax를 맞추면 기준선의 log loss가 부풀려져 잣대가 어긋난다.)
    prior = np.bincount(y[tr], minlength=3).astype(float) + 1
    prior = prior / prior.sum()
    tabular_predictions.append(prediction_frame(
        "Always flat", dates[te], y[te], np.tile(prior, (len(te), 1)), fold["fold"],
        y_pred=np.ones(len(te), dtype=int),
    ))

    logistic = make_logistic()
    logistic.fit(X[tr], y[tr])
    p_log = logistic.predict_proba(X[te])
    p_log = p_log[:, np.argsort(logistic.named_steps["model"].classes_)]
    tabular_predictions.append(prediction_frame("Logistic", dates[te], y[te], p_log, fold["fold"]))

    lgbm = make_lgbm()
    lgbm.fit(X[tr], y[tr])
    p_lgb = lgbm.predict_proba(X[te])
    p_lgb = p_lgb[:, np.argsort(lgbm.classes_)]
    tabular_predictions.append(prediction_frame("LightGBM", dates[te], y[te], p_lgb, fold["fold"]))
    lgb_importances.append(lgbm.feature_importances_)

PREDICTION_PARTS["tabular"] = pd.concat(tabular_predictions, ignore_index=True)
predictions = assemble_predictions()
display(summarize_predictions(predictions).style.format("{:.4f}", na_rep="—"))

## 6. 직접 학습하는 Transformer Encoder

30거래일의 특징 시퀀스를 입력받아 마지막 토큰에서 세 방향의 확률을 출력합니다. 각 폴드마다 scaler와 모델을 과거 데이터로만 다시 학습합니다. 작은 데이터에서 대형 Transformer는 과적합하기 쉬우므로 의도적으로 작은 구조를 사용합니다.


In [ ]:
if RUN_TRANSFORMER and TORCH_AVAILABLE:

    class DirectionTransformer(nn.Module):
        def __init__(self, n_features, seq_len, d_model=64, nhead=4, num_layers=2, dropout=0.20):
            super().__init__()
            self.seq_len = seq_len
            self.input_dropout = nn.Dropout(0.30)   # 입력 특징 단위 dropout
            self.input_proj = nn.Linear(n_features, d_model)
            # 고정 사인 위치 인코딩(학습 파라미터보다 소표본에서 안정적)
            position = torch.arange(seq_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(1, seq_len, d_model)
            pe[0, :, 0::2] = torch.sin(position * div)
            pe[0, :, 1::2] = torch.cos(position * div)
            self.register_buffer("position", pe)
            layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
                dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
            self.norm = nn.LayerNorm(d_model)
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(d_model // 2, 3),
            )

        def forward(self, x):
            z = self.input_proj(self.input_dropout(x)) + self.position[:, :x.size(1)]
            z = self.encoder(z)
            return self.head(self.norm(z[:, -1]))


    def sequences_for_indices(X_scaled, y_array, target_indices, seq_len):
        seq_x, seq_y, kept = [], [], []
        for idx in target_indices:
            if idx < seq_len - 1:
                continue
            seq_x.append(X_scaled[idx - seq_len + 1: idx + 1])
            seq_y.append(y_array[idx])
            kept.append(idx)
        if not seq_x:
            return np.empty((0, seq_len, X_scaled.shape[1])), np.array([]), np.array([])
        return np.asarray(seq_x, dtype=np.float32), np.asarray(seq_y, dtype=np.int64), np.asarray(kept)


    def _fit_epochs(X_seq, y_seq, n_features, epochs, seed, val=None):
        """지정한 epoch 수만큼 학습한다. val이 주어지면 조기종료 이력도 함께 반환."""
        torch.manual_seed(seed)          # 전역 RNG에 의존하지 않도록 매번 명시적으로 시드
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        generator = torch.Generator().manual_seed(seed)
        loader = DataLoader(
            TensorDataset(torch.from_numpy(X_seq), torch.from_numpy(y_seq)),
            batch_size=BATCH_SIZE, shuffle=True, generator=generator, drop_last=False,
        )
        model = DirectionTransformer(n_features, SEQ_LEN).to(DEVICE)
        counts = np.bincount(y_seq, minlength=3).astype(float)
        weights = len(y_seq) / (3 * np.maximum(counts, 1))
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
        optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-3)

        val_loader = None
        if val is not None:
            val_loader = DataLoader(
                TensorDataset(torch.from_numpy(val[0]), torch.from_numpy(val[1])),
                batch_size=BATCH_SIZE * 2, shuffle=False,
            )
        best_state, best_val, best_epoch = None, np.inf, epochs
        patience_left = TRANSFORMER_PATIENCE
        history = []
        for epoch in range(epochs):
            model.train()
            train_losses = []
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_losses.append(loss.item())
            train_loss = float(np.mean(train_losses))
            if val_loader is None:
                history.append((epoch + 1, train_loss, np.nan))
                continue
            model.eval()
            val_losses = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    val_losses.append(criterion(model(xb), yb).item())
            val_loss = float(np.mean(val_losses)) if val_losses else train_loss
            history.append((epoch + 1, train_loss, val_loss))
            if val_loss < best_val - 1e-4:
                best_val, best_epoch = val_loss, epoch + 1
                best_state = deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
                patience_left = TRANSFORMER_PATIENCE
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        return model, best_epoch, history


    def train_transformer(X_array, y_array, train_indices, seed=SEED, epochs=TRANSFORMER_EPOCHS):
        """조기종료로 최적 epoch를 정한 뒤, 학습 구간 전체로 다시 적합한다.
        (기존 코드는 마지막 15%를 검증에만 쓰고 학습에 넣지 않아, 실제로는 테스트
        구간보다 9개월 이른 데이터까지만 학습된 모델을 사용했다.)"""
        scaler = StandardScaler().fit(X_array[train_indices])
        X_scaled = scaler.transform(X_array).astype(np.float32)

        valid_targets = train_indices[train_indices >= SEQ_LEN - 1]
        cut = max(1, int(len(valid_targets) * 0.85))
        fit_targets, val_targets = valid_targets[:cut], valid_targets[cut:]
        if len(val_targets) < 10:
            val_targets = valid_targets[-max(10, len(valid_targets) // 10):]
            fit_targets = valid_targets[:len(valid_targets) - len(val_targets)]

        X_fit, y_fit, _ = sequences_for_indices(X_scaled, y_array, fit_targets, SEQ_LEN)
        X_val, y_val, _ = sequences_for_indices(X_scaled, y_array, val_targets, SEQ_LEN)
        _, best_epoch, history = _fit_epochs(
            X_fit, y_fit, X_array.shape[1], epochs, seed, val=(X_val, y_val))

        # 최적 epoch 수를 알았으니 검증 구간까지 포함해 다시 학습한다.
        X_all, y_all, _ = sequences_for_indices(X_scaled, y_array, valid_targets, SEQ_LEN)
        model, _, _ = _fit_epochs(X_all, y_all, X_array.shape[1], best_epoch, seed + 1)
        return model, scaler, X_scaled, history, best_epoch


    def transformer_probs(model, X_scaled, target_indices):
        dummy_y = np.zeros(len(X_scaled), dtype=np.int64)
        X_seq, _, kept = sequences_for_indices(X_scaled, dummy_y, target_indices, SEQ_LEN)
        loader = DataLoader(torch.from_numpy(X_seq), batch_size=BATCH_SIZE * 2, shuffle=False)
        probs = []
        model.eval()
        with torch.no_grad():
            for xb in loader:
                probs.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy())
        return np.vstack(probs), kept

else:
    print("RUN_TRANSFORMER=False 또는 torch 없음: Transformer 섹션을 건너뜁니다.")

In [ ]:
if RUN_TRANSFORMER and TORCH_AVAILABLE:
    transformer_predictions, transformer_histories = [], []
    for fold in tqdm(folds, desc="Transformer walk-forward"):
        tr, te = fold["train_idx"], fold["test_idx"]
        model_t, scaler_t, X_scaled_t, hist_t, best_ep = train_transformer(
            X, y, tr, seed=SEED + fold["fold"])
        probs_t, kept_t = transformer_probs(model_t, X_scaled_t, te)
        transformer_predictions.append(prediction_frame(
            "Transformer", dates[kept_t], y[kept_t], probs_t, fold["fold"]))
        transformer_histories.append(
            pd.DataFrame(hist_t, columns=["epoch", "train_loss", "val_loss"]).assign(fold=fold["fold"]))
        del model_t
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    PREDICTION_PARTS["transformer"] = pd.concat(transformer_predictions, ignore_index=True)
    predictions = assemble_predictions()
    display(summarize_predictions(predictions).style.format("{:.4f}", na_rep="—"))

    history_df = pd.concat(transformer_histories, ignore_index=True)
    sns.lineplot(data=history_df, x="epoch", y="val_loss", hue="fold", marker="o")
    plt.title("Transformer validation loss by fold")
    plt.show()
else:
    PREDICTION_PARTS["transformer"] = None
    predictions = assemble_predictions()

## 6.5 앙상블

`ENSEMBLE_MODELS`에 지정한 모델들의 확률을 **단순 평균**합니다.

이전 버전에는 OOF 스태킹 메타모델과 `exp(-2 × Δlog loss)` 성과가중 앙상블이 있었지만, 검증 결과 둘 다 제거했습니다.

- **스태킹**: 메타모델이 폴드에 따라 123행으로 학습되어 단순 평균보다 나빴습니다(log loss 1.0507 vs 1.0379).
- **성과가중**: 0.03 nats 차이 위에서 계산된 가중치라 사실상 균등 평균(0.32~0.34)이었고, 판별력을 잃고 사전확률에 가까워진 모델에 가장 큰 가중을 주는 부작용이 있었습니다.

정규화를 바로잡은 Logistic + LightGBM의 단순 평균이, 이전의 3모델 평균·스태킹보다 좋거나 같으면서 훨씬 단순합니다.

In [ ]:
# 검증 결과 스태킹 메타모델(폴드에 따라 123행으로 학습)은 단순 평균보다 나빴고
# (log loss 1.0507 vs 1.0379), exp(-2·Δlog loss) 성과가중은 0.03 nats 차이 위에서
# 계산되어 사실상 균등 평균이었다. 그래서 둘 다 제거하고 단순 평균만 남긴다.
available = [m for m in ENSEMBLE_MODELS if m in set(predictions["model"])]
if len(available) < 2:
    print("⚠️ 앙상블을 만들 기본 모델이 부족합니다:", available)
    PREDICTION_PARTS["ensemble"] = None
else:
    wide = (
        predictions[predictions["model"].isin(available)]
        .pivot_table(index="date", columns="model", values=PROB_COLS)
        .dropna()
    )
    stacked = np.stack([wide[[(p, m) for p in PROB_COLS]].to_numpy() for m in available], axis=1)
    p_mean = stacked.mean(axis=1)
    ens_dates = pd.DatetimeIndex(wide.index)
    fold_by_date = (
        predictions[predictions["model"] == available[0]]
        .set_index("date")["fold"].reindex(ens_dates)
    )
    PREDICTION_PARTS["ensemble"] = prediction_frame(
        "Mean ensemble", ens_dates,
        model_df.loc[ens_dates, "target"].to_numpy(dtype=int),
        p_mean, fold_by_date.to_numpy(),
    )
    print(f"Mean ensemble = {' + '.join(available)} 확률의 단순 평균 ({len(ens_dates)}일)")

predictions = assemble_predictions()

native_metrics = summarize_predictions(predictions, with_decomposition=True)
display(native_metrics[["n", "accuracy", "balanced_accuracy", "macro_f1", "log_loss", "brier"]]
        .style.format("{:.4f}", na_rep="—"))

# 폴드별 성능: 우위가 여러 구간에서 반복되는지 확인
fold_table = (
    predictions.groupby(["fold", "model"], group_keys=False)[["y_true", "y_pred", *PROB_COLS]]
    .apply(lambda g: pd.Series(metric_row(g)))
    .reset_index()
    .pivot(index="fold", columns="model", values="balanced_accuracy")
)
cols = [m for m in MODEL_ORDER if m in fold_table.columns]
display(fold_table[cols].style.format("{:.3f}", na_rep="—")
        .set_caption("Balanced accuracy by walk-forward fold"))

## 7. 금융 시계열 파운데이션 모델 Kronos-small (기본 꺼짐)

Kronos는 OHLCV를 금융시장 전용 토큰으로 변환한 뒤 다음 K-line을 생성합니다. 삼성전자에 별도 학습하지 않은 **zero-shot** 성능을 확인합니다.

**기본값이 `RUN_KRONOS = False`인 이유**: 60일 평가에서 log loss 1.87 / Brier 1.07로 "항상 보합" 기준선(1.10 / 0.67)보다 나빴고, 이미 앙상블에서도 제외되어 있었습니다. 반면 이 섹션 하나 때문에 git clone + HuggingFace 가중치라는 무거운 의존성이 붙습니다.

켜서 실행하려면 `RUN_KRONOS = True`로 두되, 다음을 유의하세요.

- 8샘플 Monte Carlo 빈도를 확률로 쓰므로 log loss는 표본 잡음에 지배됩니다.
- 60일 평가로는 예측력의 유무를 판정할 수 없습니다(무작위 예측의 60일 balanced accuracy 표준편차가 약 0.065). 최소 250일 이상을 권장합니다.

In [ ]:
kronos_predictor = None
kronos_predictions = pd.DataFrame()

if RUN_KRONOS:
    if not TORCH_AVAILABLE:
        print("torch가 없어 Kronos를 건너뜁니다.")
        RUN_KRONOS = False
    else:
        try:
            import subprocess
            subprocess.run(
                "pip -q install einops==0.8.1 huggingface_hub==0.33.1 tqdm==4.67.1 safetensors==0.6.2",
                shell=True, check=False,
            )
            if not Path("/content/Kronos").exists():
                subprocess.run(
                    "git clone --depth 1 https://github.com/shiyu-coder/Kronos.git /content/Kronos",
                    shell=True, check=False,
                )
            if "/content/Kronos" not in sys.path:
                sys.path.append("/content/Kronos")
            from model import Kronos, KronosTokenizer, KronosPredictor

            kronos_device = "cuda:0" if torch.cuda.is_available() else "cpu"
            tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
            kronos_model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
            kronos_predictor = KronosPredictor(kronos_model, tokenizer, device=kronos_device, max_context=512)
            print("Kronos loaded on", kronos_device)
        except Exception as exc:  # ImportError만으로는 부족(HF 레이트리밋/레포 변경 등)
            print("⚠️ Kronos 로드 실패, 건너뜁니다:", exc)
            RUN_KRONOS = False
else:
    print("RUN_KRONOS=False: Kronos 섹션을 건너뜁니다. "
          "(zero-shot 성능이 '항상 보합' 기준선보다 나빴고, 무거운 의존성의 유일한 원인입니다.)")

In [ ]:
def kronos_probability_for_date(target_date, mc_samples=KRONOS_MC_SAMPLES):
    target_date = pd.Timestamp(target_date).normalize()
    hist = sam.loc[sam.index < target_date].tail(KRONOS_LOOKBACK).copy()
    if len(hist) < 100:
        raise ValueError("Kronos context is too short")

    band_value = float(model_df.loc[target_date, "band"]) if target_date in model_df.index else live_band

    x_df = hist[["open", "high", "low", "close", "volume"]].copy()
    x_df["volume"] = x_df["volume"].fillna(0).clip(lower=0)
    x_timestamp = pd.Series(hist.index)
    y_timestamp = pd.Series([target_date])
    previous_close = float(hist["close"].iloc[-1])
    if TARGET_MODE == "open_to_close":
        if target_date in sam.index:
            reference_price = float(sam.loc[target_date, "open"])
        else:
            reference_price = float(LIVE_OPEN_PRICE)
    else:
        reference_price = previous_close

    # 전역 RNG를 오염시키지 않도록 상태를 저장했다가 복원한다.
    py_state, np_state = random.getstate(), np.random.get_state()
    torch_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        sampled_returns = []
        for sample_no in range(mc_samples):
            sample_seed = SEED + sample_no
            random.seed(sample_seed)
            np.random.seed(sample_seed)
            torch.manual_seed(sample_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(sample_seed)
            pred_df = kronos_predictor.predict(
                df=x_df, x_timestamp=x_timestamp, y_timestamp=y_timestamp,
                pred_len=1, T=0.8, top_p=0.9, sample_count=1,
            )
            sampled_returns.append(float(pred_df["close"].iloc[0]) / reference_price - 1)
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.set_rng_state(torch_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)

    sampled_returns = np.asarray(sampled_returns)
    counts = np.array([
        np.sum(sampled_returns < -band_value),
        np.sum((sampled_returns >= -band_value) & (sampled_returns <= band_value)),
        np.sum(sampled_returns > band_value),
    ], dtype=float)
    probs = (counts + 0.5) / (counts.sum() + 1.5)   # Jeffreys smoothing
    return probs, sampled_returns


if RUN_KRONOS and kronos_predictor is not None:
    if KRONOS_EVAL_DAYS < 250:
        print(f"⚠️ KRONOS_EVAL_DAYS={KRONOS_EVAL_DAYS}일로는 예측력의 유무를 판정할 수 없습니다"
              f" (무작위 예측의 60일 balanced accuracy 표준편차가 약 0.065). 참고용으로만 보세요.")
    all_test_dates = pd.DatetimeIndex(sorted(predictions["date"].unique()))
    kronos_dates = all_test_dates.intersection(sam.index)[-KRONOS_EVAL_DAYS:]
    rows = []
    for target_date in tqdm(kronos_dates, desc="Kronos zero-shot backtest"):
        try:
            probs_k, sampled_ret = kronos_probability_for_date(target_date)
            rows.append(prediction_frame(
                "Kronos-small", [target_date], [int(model_df.loc[target_date, "target"])],
                probs_k.reshape(1, -1), fold_id="zero-shot",
                extra={"pred_return_mean": [float(sampled_ret.mean())],
                       "pred_return_std": [float(sampled_ret.std(ddof=0))]},
            ))
        except Exception as exc:
            print(f"⚠️ {target_date.date()} Kronos 실패: {exc}")
    PREDICTION_PARTS["kronos"] = pd.concat(rows, ignore_index=True) if rows else None
else:
    PREDICTION_PARTS["kronos"] = None

predictions = assemble_predictions()
if PREDICTION_PARTS.get("kronos") is not None:
    display(summarize_predictions(PREDICTION_PARTS["kronos"], with_ci=False).style.format("{:.4f}", na_rep="—"))

## 8. 성능 표와 갭/세션 분해

세 종류의 표를 봅니다.

1. **전체 평가 구간 성능** — 모든 지표에 월 블록 부트스트랩 95% 신뢰구간을 붙였습니다. 일별 예측은 서로 독립이 아니므로 달력 월을 블록으로 복원추출합니다.
2. **갭 / 세션 분해** — 이 노트북에서 가장 중요한 표입니다. 모델의 방향 점수가 *전일 종가→시가 갭*의 부호를 맞히는지, *시가→종가 세션*의 부호를 맞히는지 따로 측정하고, 각 구간에서 모델 방향대로 포지션을 잡았을 때의 일평균 손익(bp)을 왕복 비용 차감 전후로 보여줍니다. **07:00 예측으로 실제 취할 수 있는 것은 세션 구간뿐입니다.**
3. **'항상 보합' 대비 쌍체 차이** — 같은 날짜에서 계산한 차이와 신뢰구간입니다. CI가 0을 포함하면 "동률"로 읽어야 합니다.

Kronos를 켰을 때만 공통 날짜 참고표가 추가됩니다. 그 표본(기본 60일)에서는 무작위 예측도 약 49%의 확률로 1/3 기준선을 넘으므로 순위를 읽으면 안 됩니다.

In [ ]:
native_metrics = summarize_predictions(predictions, with_decomposition=True)

print("■ 전체 평가 구간 성능 (95% 신뢰구간은 달력 월 블록 부트스트랩)")
display(
    native_metrics[["n", "accuracy", "balanced_accuracy", "bal_acc_lo", "bal_acc_hi",
                    "macro_f1", "log_loss", "log_loss_lo", "log_loss_hi", "brier"]]
    .style.format("{:.4f}", na_rep="—")
)

# ---- 이 모델이 실제로 무엇을 예측하는가 --------------------------------------
print("\n■ 갭 / 세션 분해 — 07:00 예측으로 실제 취할 수 있는 것은 '세션'뿐이다")
print("   갭     = 전일 종가 → 당일 시가 (09:00에 이미 가격에 반영됨)")
print("   세션   = 당일 시가 → 당일 종가 (시가에 진입하면 취할 수 있는 유일한 구간)")
decomp_cols = ["auc_gap", "auc_session", "gap_bp", "session_bp", "session_bp_lo",
               "session_bp_hi", "session_bp_net"]
display(
    native_metrics[decomp_cols].style.format(
        {"auc_gap": "{:.3f}", "auc_session": "{:.3f}", "gap_bp": "{:+.1f}",
         "session_bp": "{:+.1f}", "session_bp_lo": "{:+.1f}", "session_bp_hi": "{:+.1f}",
         "session_bp_net": "{:+.1f}"}, na_rep="—")
    .set_caption(f"bp = 일평균 수익(베이시스포인트). net은 왕복 비용 {COST_BP:.0f}bp 차감 후.")
)

# ---- 차이가 실재하는가: 쌍체 비교 --------------------------------------------
reference = "Always flat"
compare = [m for m in MODEL_ORDER if m in set(predictions["model"]) and m != reference]
rows = []
for name in compare:
    for metric in ("balanced_accuracy", "log_loss"):
        d = paired_delta_ci(predictions, name, reference, metric=metric)
        rows.append({
            "model": name, "metric": metric, "n": d["n"],
            "delta_vs_flat": d["delta"], "lo": d["lo"], "hi": d["hi"],
            "판정": "유의" if (pd.notna(d["lo"]) and (d["lo"] > 0) == (d["hi"] > 0)) else "동률(CI가 0 포함)",
        })
    rows.append({"model": name, "metric": "folds_worse_than_prior",
                 "n": len(folds), "delta_vs_flat": folds_worse_than_prior(predictions, name),
                 "lo": np.nan, "hi": np.nan, "판정": ""})
print("\n■ '항상 보합' 대비 쌍체 차이 (CI가 0을 포함하면 동률로 읽어야 한다)")
display(pd.DataFrame(rows).set_index(["model", "metric"])
        .style.format({"delta_vs_flat": "{:+.4f}", "lo": "{:+.4f}", "hi": "{:+.4f}"}, na_rep="—"))

# ---- Kronos가 있을 때만: 공통 날짜 참고표 -----------------------------------
common_metrics = None
if PREDICTION_PARTS.get("kronos") is not None:
    model_date_sets = {n: set(pd.to_datetime(g["date"])) for n, g in predictions.groupby("model")}
    common_dates = set.intersection(*model_date_sets.values())
    if len(common_dates) >= 10:
        common_metrics = summarize_predictions(
            predictions[predictions["date"].isin(common_dates)], with_ci=False)
        print(f"\n■ [참고] 모든 모델이 예측한 공통 날짜 {len(common_dates)}일")
        print("   ⚠️ 이 표본에서는 무작위 예측도 약 49%의 확률로 1/3 기준선을 넘습니다. 순위를 읽지 마세요.")
        display(common_metrics.style.format("{:.4f}", na_rep="—"))

In [ ]:
# 그림은 항상 전체 평가 구간에서 그린다(60일 창은 검정력이 없다).
plot_metrics = native_metrics.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

axes[0].barh(plot_metrics["model"], plot_metrics["balanced_accuracy"], color="#4c78a8")
axes[0].errorbar(
    plot_metrics["balanced_accuracy"], plot_metrics["model"],
    xerr=[plot_metrics["balanced_accuracy"] - plot_metrics["bal_acc_lo"],
          plot_metrics["bal_acc_hi"] - plot_metrics["balanced_accuracy"]],
    fmt="none", ecolor="black", capsize=3, lw=1,
)
axes[0].axvline(1 / 3, color="black", ls="--", lw=1, label="random 3-class")
axes[0].set_title("Balanced accuracy (95% CI) ↑")
axes[0].legend()

axes[1].barh(plot_metrics["model"], plot_metrics["log_loss"], color="#e15759")
axes[1].errorbar(
    plot_metrics["log_loss"], plot_metrics["model"],
    xerr=[plot_metrics["log_loss"] - plot_metrics["log_loss_lo"],
          plot_metrics["log_loss_hi"] - plot_metrics["log_loss"]],
    fmt="none", ecolor="black", capsize=3, lw=1,
)
axes[1].set_title("Log loss (95% CI) ↓")
axes[1].set_xlim(left=min(0.95, plot_metrics["log_loss_lo"].min() - 0.02))

width = 0.38
positions = np.arange(len(plot_metrics))
axes[2].barh(positions + width / 2, plot_metrics["auc_gap"], height=width,
             color="#4c78a8", label="갭 (거래 불가)")
axes[2].barh(positions - width / 2, plot_metrics["auc_session"], height=width,
             color="#f28e2b", label="세션 (거래 가능)")
axes[2].set_yticks(positions)
axes[2].set_yticklabels(plot_metrics["model"])
axes[2].axvline(0.5, color="black", ls="--", lw=1)
axes[2].set_xlim(0.35, 0.9)
axes[2].set_title("방향 판별 AUC: 갭 vs 세션")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
models_to_plot = [m for m in MODEL_ORDER if m in set(predictions["model"])]
ncols = 3
nrows = math.ceil(len(models_to_plot) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, name in zip(axes, models_to_plot):
    group = predictions[predictions["model"] == name]
    cm = confusion_matrix(group["y_true"], group["y_pred"], labels=[0, 1, 2], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=["down", "flat", "up"], yticklabels=["down", "flat", "up"], ax=ax)
    ax.set_title(f"{name} (n={len(group)})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
for ax in axes[len(models_to_plot):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def reliability_by_class(frame, class_index, bins=10):
    """클래스별 예측확률을 분위로 나눠 실제 발생률과 비교한다.
    (최대확률 기준 구간화는 3-class에서 (0,1/3) 구간이 구조적으로 비어 버린다.)"""
    work = frame.copy()
    p = work[PROB_COLS[class_index]].to_numpy()
    hit = (work["y_true"].to_numpy() == class_index).astype(int)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return pd.DataFrame()
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, len(edges) - 2)
    out = pd.DataFrame({"bin": idx, "p": p, "hit": hit}).groupby("bin").agg(
        n=("hit", "size"), mean_p=("p", "mean"), observed=("hit", "mean"))
    return out[out["n"] >= 20]


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for class_index, ax in enumerate(axes):
    for name in models_to_plot:
        group = predictions[predictions["model"] == name]
        if len(group) < 200:
            continue
        rel = reliability_by_class(group, class_index)
        if len(rel):
            ax.plot(rel["mean_p"], rel["observed"], marker="o", ms=4, label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim(0, 0.8)
    ax.set_ylim(0, 0.8)
    ax.set_title(f"P({LABEL_NAMES[class_index]}) 신뢰도")
    ax.set_xlabel("예측 확률")
    ax.set_ylabel("실제 발생률")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 9. LightGBM 변수 중요도

중요도는 인과관계가 아니라 분할에 자주 사용된 정도입니다. 여러 폴드의 중요도를 평균합니다.


In [ ]:
if lgb_importances:
    importance = pd.Series(np.mean(lgb_importances, axis=0), index=feature_cols).sort_values(ascending=False)
    display(importance.head(20).to_frame("mean_split_importance"))
    importance.head(20).sort_values().plot.barh(figsize=(8, 7), color="#4c78a8")
    plt.title("LightGBM mean feature importance")
    plt.show()


## 10. 전체 데이터로 재학습하고 다음 거래일 방향·중기 가격 예측

아래 결과는 노트북 실행 시점의 최신 공개 데이터로 계산됩니다. 기존 다음 거래일 방향과 함께 5거래일(1주일), 20거래일(1개월) 뒤 종가를 직접 회귀 방식으로 예측합니다. 한국 휴일이 끼어 예측 날짜가 틀리면 설정 셀의 `PREDICTION_DATE_OVERRIDE`를 수정하고 특징 생성 이후 셀을 다시 실행하세요.


In [ ]:
live_X = live_row[feature_cols].to_numpy(dtype=np.float32)
live_probs = {}

logistic_full = make_logistic().fit(X, y)
p = logistic_full.predict_proba(live_X)
live_probs["Logistic"] = p[:, np.argsort(logistic_full.named_steps["model"].classes_)][0]

lgbm_full = make_lgbm().fit(X, y)
p = lgbm_full.predict_proba(live_X)
live_probs["LightGBM"] = p[:, np.argsort(lgbm_full.classes_)][0]

transformer_full = transformer_scaler = None
if RUN_TRANSFORMER and TORCH_AVAILABLE:
    all_train_idx = np.arange(len(X))
    transformer_full, transformer_scaler, _, transformer_full_history, _ = train_transformer(
        X, y, all_train_idx, seed=SEED + 1000)
    X_plus_live_scaled = transformer_scaler.transform(np.vstack([X, live_X])).astype(np.float32)
    live_seq = torch.from_numpy(X_plus_live_scaled[-SEQ_LEN:][None, ...]).to(DEVICE)
    transformer_full.eval()
    with torch.no_grad():
        live_probs["Transformer"] = torch.softmax(transformer_full(live_seq), dim=1).cpu().numpy()[0]

if RUN_KRONOS and kronos_predictor is not None:
    try:
        p_kronos_live, kronos_live_returns = kronos_probability_for_date(prediction_date)
        live_probs["Kronos-small"] = p_kronos_live
    except Exception as exc:
        print("⚠️ Latest Kronos forecast failed:", exc)

# 앙상블은 ENSEMBLE_MODELS의 단순 평균 하나만 쓴다.
# (성과가중 exp(-2·Δlog loss)는 0.03 nats 차이 위에서 계산되어 사실상 균등 평균이었고,
#  판별력을 잃고 사전확률에 가까워진 모델에 가장 큰 가중을 주는 부작용이 있었다.)
ensemble_members = [m for m in ENSEMBLE_MODELS if m in live_probs]
if len(ensemble_members) >= 2:
    live_probs["Mean ensemble"] = np.mean([live_probs[m] for m in ensemble_members], axis=0)
ensemble_weights = {m: 1.0 / len(ensemble_members) for m in ensemble_members}

live_table = []
for name, probs in live_probs.items():
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum()
    live_table.append({
        "model": name,
        "prediction_date": prediction_date.date().isoformat(),
        "prediction": LABEL_NAMES[int(np.argmax(probs))],
        "p_down": probs[0], "p_flat": probs[1], "p_up": probs[2],
    })
live_table = pd.DataFrame(live_table).set_index("model")

print("앙상블 구성:", ensemble_members, "(단순 평균)")
if imputed_live_features:
    print("⚠️ 과거값으로 대체된 라이브 특징:", imputed_live_features)
print("⚠️ 이 예측의 타깃은", TARGET_MODE, "입니다.",
      "close_to_close라면 예측력의 대부분은 09:00 시가에 이미 반영되는 '갭'이며,"
      " 시가 진입 후 종가 청산으로 얻을 수 있는 몫은 8절의 session_bp 열을 보세요."
      if TARGET_MODE == "close_to_close" else "")
display(live_table.style.format({"p_down": "{:.1%}", "p_flat": "{:.1%}", "p_up": "{:.1%}"}))

### 10.1 1주일·1개월 뒤 예상 종가

각 시점의 미래 수익률을 **변동성 스케일 타깃 + 강한 축소(Ridge)** 로 추정합니다. 이전 버전에서 바뀐 점은 다음과 같습니다.

- 검증에서 두 회귀 모델(Ridge, LightGBM) 모두 **"수익률 0% 예측" 기준선보다 나빴는데도** 점 예측이 그대로 보고되었습니다. 이제는 기준선을 이기지 못하면 예상 수익률을 **0%로 두고** `signal` 열에 "없음"으로 표시합니다.
- 예측값은 OOF에서 추정한 **축소계수**(실제 수익률을 예측값에 회귀시킨 기울기, 0~1)를 곱한 값입니다. 축소가 없으면 20일 지평에서 예측 분산이 근거보다 10배 이상 컸습니다.
- 단일 예상 종가 대신 **변동성 스케일 구간**(`low_close` ~ `high_close`)을 함께 제공하고, 실제 달성 커버리지를 `band_coverage`에 적습니다.
- LightGBM 회귀는 OOF 우위가 없고 시드에 따라 라이브 값이 몇 %p씩 흔들려 제거했습니다.
- `target_date`는 KRX 거래일 달력 기준입니다(`exchange_calendars`가 없으면 주말만 제외하고 경고합니다).

In [ ]:
FORECAST_HORIZONS = {"1주일": 5, "1개월": 20}
BAND_COVERAGE = 0.80        # 예측 구간의 목표 커버리지


def make_price_model():
    # 강한 축소(Ridge alpha 큼) + 변동성 스케일 타깃. LightGBM 회귀는 OOF에서 우위가 없고
    # 시드에 따라 라이브 값이 몇 %p씩 움직여 제거했다.
    return Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1e4))])


def fit_price_forecast(horizon, live_features):
    """행 d의 특징은 d-1 종가까지만 포함한다. 목표값은 d-1 종가 대비
    horizon번째 거래일(d+horizon-1)의 원본 종가 수익률이다.
    (표시용 current_close와 같은 가격 계열을 써야 예상 종가가 일관된다.)"""
    future_return = sam_raw_close.shift(-(horizon - 1)) / sam_raw_close.shift(1) - 1
    reg = feat.loc[sam.index, feature_cols].copy()
    reg["future_return"] = future_return.reindex(reg.index)
    reg["sigma_h"] = (feat.loc[sam.index, "sam_vol_20"] * np.sqrt(horizon))
    reg = reg.replace([np.inf, -np.inf], np.nan).dropna()

    X_reg = reg[feature_cols].to_numpy(dtype=np.float32)
    y_reg = reg["future_return"].to_numpy(dtype=np.float64)
    sigma_h = reg["sigma_h"].to_numpy(dtype=np.float64)
    z = y_reg / np.maximum(sigma_h, 1e-6)

    splitter = TimeSeriesSplit(n_splits=3 if QUICK_MODE else 5, gap=horizon - 1)
    template = make_price_model()
    oof = np.full(len(z), np.nan)
    for train_idx, valid_idx in splitter.split(X_reg):
        fold_model = clone(template).fit(X_reg[train_idx], z[train_idx])
        oof[valid_idx] = fold_model.predict(X_reg[valid_idx]) * sigma_h[valid_idx]

    mask = ~np.isnan(oof)
    zero_mae = float(np.mean(np.abs(y_reg[mask])))
    raw_mae = float(np.mean(np.abs(y_reg[mask] - oof[mask])))

    # 축소계수: OOF에서 실제 수익률을 예측값에 회귀시킨 기울기(0~1로 제한).
    denom = float(np.sum(oof[mask] ** 2))
    slope = float(np.clip(np.sum(oof[mask] * y_reg[mask]) / denom, 0.0, 1.0)) if denom > 0 else 0.0
    shrunk_mae = float(np.mean(np.abs(y_reg[mask] - slope * oof[mask])))

    # 기준선을 "우연이 아니라고 할 만큼" 이겨야 신호로 인정한다.
    # 점추정치만 비교하면 0.0002 차이도 통과해 버린다. 겹치는 라벨의 자기상관을
    # 감안해 달력 월 블록 부트스트랩으로 손실 차이의 신뢰구간을 구한다.
    loss_diff = np.abs(y_reg[mask] - slope * oof[mask]) - np.abs(y_reg[mask])
    diff_dates = reg.index[mask]
    diff_lo, diff_hi = block_bootstrap_ci(diff_dates, lambda idx: float(np.mean(loss_diff[idx])))
    beats_baseline = bool(pd.notna(diff_hi) and diff_hi < 0)
    if not beats_baseline:
        slope = 0.0
        shrunk_mae = zero_mae

    # 구간: |실제 - 중심| / sigma_h 의 경험 분위수(명목 커버리지에 맞춰 보정).
    resid_scaled = np.abs(y_reg[mask] - slope * oof[mask]) / np.maximum(sigma_h[mask], 1e-6)
    q = float(np.quantile(resid_scaled, BAND_COVERAGE))
    realized_coverage = float(np.mean(resid_scaled <= q))

    fitted = clone(template).fit(X_reg, z)
    live_sigma_h = float(live_row["sam_vol_20"].iloc[0]) * np.sqrt(horizon)
    point = float(slope * fitted.predict(live_features)[0] * live_sigma_h)

    stats = {
        "zero_baseline_mae": zero_mae,
        "raw_model_mae": raw_mae,
        "shrunk_model_mae": shrunk_mae,
        "mae_diff_vs_zero": float(np.mean(loss_diff)),
        "mae_diff_lo": float(diff_lo) if pd.notna(diff_lo) else None,
        "mae_diff_hi": float(diff_hi) if pd.notna(diff_hi) else None,
        "oof_slope": slope,
        "beats_baseline": bool(beats_baseline),
        "band_q": q,
        "band_coverage_realized": realized_coverage,
        "live_sigma_h": live_sigma_h,
        "n_oof": int(mask.sum()),
    }
    return point, live_sigma_h * q, stats, fitted


current_close = float(sam_raw_close.iloc[-1])
price_forecast_rows = []
price_forecast_stats = {}
price_forecast_models = {}

for horizon_label, horizon in FORECAST_HORIZONS.items():
    point, half_width, stats, fitted = fit_price_forecast(horizon, live_X)
    price_forecast_stats[horizon_label] = stats
    price_forecast_models[horizon_label] = fitted
    target_date = krx_sessions_ahead(prediction_date, horizon)
    if not stats["beats_baseline"]:
        print(f"ℹ️ horizon={horizon}: 모델 MAE {stats['raw_model_mae']:.4f} vs "
              f"0% 기준선 {stats['zero_baseline_mae']:.4f}, 차이 "
              f"{stats['mae_diff_vs_zero']:+.4f} "
              f"[{stats['mae_diff_lo']:+.4f}, {stats['mae_diff_hi']:+.4f}] — "
              f"신뢰구간이 0을 포함하므로 예상 수익률을 0%로 둡니다(정보 없음).")
    price_forecast_rows.append({
        "horizon": horizon_label,
        "trading_days": horizon,
        "as_of_date": last_samsung_date.date().isoformat(),
        "target_date": target_date.date().isoformat(),
        "current_close": current_close,
        "signal": "있음" if stats["beats_baseline"] else "없음 (0% 기준선 미달)",
        "predicted_return": point,
        "predicted_close": current_close * (1 + point),
        "low_close": current_close * (1 + point - half_width),
        "high_close": current_close * (1 + point + half_width),
        "band_coverage": stats["band_coverage_realized"],
        "model_mae": stats["shrunk_model_mae"],
        "zero_baseline_mae": stats["zero_baseline_mae"],
        "oof_slope": stats["oof_slope"],
    })

price_forecast_table = pd.DataFrame(price_forecast_rows).set_index("horizon")
print("\n중기 가격 예측")
print("※ 점 예측은 OOF 기울기로 축소한 값이며, 기준선을 이기지 못하면 0%로 둡니다.")
print("※ 구간은 변동성 스케일 밴드이며 target_date는 KRX 거래일 기준입니다.")
display(price_forecast_table.style.format({
    "current_close": "{:,.0f}원", "predicted_return": "{:+.2%}",
    "predicted_close": "{:,.0f}원", "low_close": "{:,.0f}원", "high_close": "{:,.0f}원",
    "band_coverage": "{:.1%}", "model_mae": "{:.2%}", "zero_baseline_mae": "{:.2%}",
    "oof_slope": "{:.3f}",
}))

## 11. 결과 저장

백테스트 예측, 공통기간 성능, 다음 거래일 방향, 1주일·1개월 가격 예측, 학습 모델과 설정을 `/content/samsung_direction_outputs.zip`으로 묶습니다. 마지막 줄의 다운로드 코드는 주석을 해제해 사용하세요.


In [ ]:
OUTPUT_DIR = Path("/content/samsung_direction_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = f"{prediction_date.date().isoformat()}_{DATA_SNAPSHOT_HASH}"

predictions.sort_values(["date", "model"]).to_csv(OUTPUT_DIR / "backtest_predictions.csv", index=False)
native_metrics.to_csv(OUTPUT_DIR / "metrics_native_window.csv")
if common_metrics is not None:
    common_metrics.to_csv(OUTPUT_DIR / "metrics_common_dates.csv")
live_table.to_csv(OUTPUT_DIR / "latest_forecast.csv")
price_forecast_table.to_csv(OUTPUT_DIR / "multi_horizon_price_forecast.csv")
pd.Series(feature_cols, name="feature").to_csv(OUTPUT_DIR / "feature_list.csv", index=False)

# ---- 예측 로그: 무엇을 예측했는지 append하고, 나중에 실제값으로 채점한다 ------
forecast_log_path = OUTPUT_DIR / "forecast_log.csv"
log_rows = live_table.reset_index().assign(
    run_id=RUN_ID,
    data_snapshot_hash=DATA_SNAPSHOT_HASH,
    as_of_date=last_samsung_date.date().isoformat(),
    target_mode=TARGET_MODE,
    band=live_band,
    imputed_features=json.dumps(imputed_live_features, ensure_ascii=False),
)
if forecast_log_path.exists():
    log_rows = pd.concat([pd.read_csv(forecast_log_path), log_rows], ignore_index=True)
    log_rows = log_rows.drop_duplicates(subset=["run_id", "model"], keep="last")
log_rows.to_csv(forecast_log_path, index=False)


def score_forecast_log(path=forecast_log_path):
    """과거 예측 로그를 실제 라벨과 대조한다(예측일이 지난 뒤 실행)."""
    if not Path(path).exists():
        return pd.DataFrame()
    log = pd.read_csv(path, parse_dates=["prediction_date"])
    truth = model_df[["target"]].rename(columns={"target": "y_true"})
    joined = log.join(truth, on="prediction_date", how="inner")
    if joined.empty:
        return joined
    joined["y_pred"] = joined[PROB_COLS].to_numpy().argmax(axis=1)
    joined["correct"] = (joined["y_pred"] == joined["y_true"]).astype(int)
    return joined.groupby("model").agg(n=("correct", "size"), accuracy=("correct", "mean"))


scored = score_forecast_log()
if len(scored):
    print("■ 과거 예측 실적 (forecast_log.csv 기준)")
    display(scored)

config = {
    "run_id": RUN_ID,
    "data_snapshot_hash": DATA_SNAPSHOT_HASH,
    "versions": VERSIONS,
    "assets": ASSETS,
    "start_date": START_DATE,
    "neutral_band": NEUTRAL_BAND,
    "target_mode": TARGET_MODE,
    "band_mode": BAND_MODE,
    "vol_band_mult": VOL_BAND_MULT,
    "live_band": live_band,
    "ensemble_models": ENSEMBLE_MODELS,
    "ensemble_weights": {k: float(v) for k, v in ensemble_weights.items()},
    "first_test_date": FIRST_TEST_DATE,
    "test_months": TEST_MONTHS,
    "rolling_train_years": ROLLING_TRAIN_YEARS,
    "quick_mode": QUICK_MODE,
    "seq_len": SEQ_LEN,
    "run_transformer": RUN_TRANSFORMER,
    "transformer_epochs": TRANSFORMER_EPOCHS,
    "run_kronos": RUN_KRONOS,
    "kronos_eval_days": KRONOS_EVAL_DAYS,
    "kronos_mc_samples": KRONOS_MC_SAMPLES,
    "cost_bp": COST_BP,
    "bootstrap_b": BOOTSTRAP_B,
    "last_samsung_date": last_samsung_date.date().isoformat(),
    "prediction_date": prediction_date.date().isoformat(),
    "model_rows": int(len(model_df)),
    "backtest_range": [model_df.index.min().date().isoformat(), model_df.index.max().date().isoformat()],
    "n_folds": len(folds),
    "imputed_live_features": imputed_live_features,
    "forecast_horizons": FORECAST_HORIZONS,
    "price_forecast_stats": price_forecast_stats,
    "asset_last_bar": {n: f.index.max().date().isoformat() for n, f in raw.items()},
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

joblib.dump(logistic_full, OUTPUT_DIR / "logistic_model.joblib")
joblib.dump(lgbm_full, OUTPUT_DIR / "lightgbm_model.joblib")
for horizon_label, fitted_model in price_forecast_models.items():
    joblib.dump(fitted_model, OUTPUT_DIR / f"price_{FORECAST_HORIZONS[horizon_label]}d_ridge.joblib")
if RUN_TRANSFORMER and TORCH_AVAILABLE and transformer_full is not None:
    joblib.dump(transformer_scaler, OUTPUT_DIR / "transformer_scaler.joblib")
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in transformer_full.state_dict().items()},
        "n_features": len(feature_cols), "seq_len": SEQ_LEN,
        "feature_cols": feature_cols, "version": 2,
    }, OUTPUT_DIR / "transformer_model.pt")

import shutil
zip_path = shutil.make_archive("/content/samsung_direction_outputs", "zip", OUTPUT_DIR)
print("Saved:", zip_path, "| run_id:", RUN_ID)

# 필요할 때 아래 두 줄의 주석을 해제하세요.
# from google.colab import files
# files.download(zip_path)

## 12. 결과 해석 체크리스트

1. **갭/세션 분해 표**의 `auc_session`이 0.5보다 의미 있게 큰가? `session_bp_net`(비용 차감 후)이 0보다 큰가? 그렇지 않다면 이 모델로 거래할 수 있는 예측력은 없다.
2. `balanced_accuracy`의 신뢰구간이 "항상 보합"의 값을 배제하는가? 점추정치만 보고 판단하지 말 것.
3. 쌍체 비교표에서 **CI가 0을 포함하는 차이는 전부 "동률"** 이다. 시드를 바꾸면 순위가 뒤집힌다.
4. `folds_worse_than_prior`가 몇 폴드인가? 절반 이상이면 그 모델은 확률 모델로서 클래스 빈도보다 못하다.
5. 하락·보합·상승 중 한 클래스만 잘 맞히는 것은 아닌가(혼동행렬 확인)?
6. 중기 가격 예측의 `signal` 열이 "없음"이면 예상 종가는 현재가와 같다는 뜻이다. 구간(`low_close`~`high_close`) 폭을 반드시 함께 보라.
7. `balanced_accuracy`는 `VOL_BAND_MULT`에 따라 단조 증가한다. **이 지표로 밴드를 튜닝하지 말 것.**
8. 라이브 예측에 "과거값으로 대체된 특징" 경고가 떴는가? 떴다면 그 예측은 신뢰하지 말고 데이터부터 고쳐라.

### 다음 개선 순서

- **09:00 제품으로 전환**: `TARGET_MODE="open_to_close"` + 09:00 시가(`LIVE_OPEN_PRICE`) 필수 입력. 일간 종가 데이터로는 세션 구간에 정보가 거의 없음이 확인되었으므로, 08:50 예상체결가·호가 잔량 같은 장직전 정보가 유일한 돌파구다.
- KRX 투자자별 수급(외국인·기관 순매수, `pykrx`로 d+1 시점 사용), VKOSPI, 실적·배당 캘린더 추가.
- 폴드 0–5에서 모든 재량 선택(밴드 배수, `C`, `num_leaves`, 피처 목록)을 동결하고 6–10을 사전등록 홀드아웃으로 보고.
- `src/predict_stock/` 패키지 + CLI(`backtest / predict / score`)로 분리해 매일 예측·채점을 자동화.
- 최소 월 1회 재학습, `forecast_log.csv`로 매일 예측과 실제 결과를 대조.

### 하지 말 것 (측정 결과 효과가 없거나 잡음 이내)

- 시드 배깅(LightGBM 5시드 평균: bal. acc +0.004 [−0.010, +0.016])
- 일간 야후 종가에서 파생되는 값싼 피처 추가(14종 합계 +0.011, 표준오차 0.014)
- `VOL_BAND_MULT`를 balanced accuracy로 튜닝
- Transformer 확대나 Kronos 파인튜닝(모든 피처의 lag-1·lag-2를 더하면 선형 모델도 나빠진다 = 시퀀스 신호가 없다)

### 참고 자료

- [Kronos 논문](https://arxiv.org/abs/2508.02739) · [공식 구현](https://github.com/shiyu-coder/Kronos)
- [KRX Data Marketplace](https://data.krx.co.kr/)